# LLM Safety & Response Evaluation Benchmark — Exploration

This notebook walks through the benchmark: what each dimension measures, how
scoring works, a single live example call, and interpretation of the full
results produced by `python -m src.runner` + `python -m src.report`.

**Setup reminder:** copy `.env.example` to `.env` and fill in whichever free
API key(s) you have (Groq is fastest to get). Any provider without a key runs
in mock mode, so this notebook works even before you've added keys — you just
won't get real scores yet.


In [ ]:
import sys, os, json
sys.path.append("..")
os.chdir("..")  # so relative paths (data/, src/, results/) resolve correctly

from dotenv import load_dotenv
load_dotenv()

from src.rubric import DIMENSIONS, composite_score
from src.models import ModelClient
from src.scorers import score_test_case
import pandas as pd


## 1. The rubric

Nine dimensions, each scored 1-5, combined into a weighted composite score.
Refusal quality and prompt-injection resistance carry the most weight since
those are the safety-critical dimensions for a deployed system.


In [ ]:
rubric_df = pd.DataFrame(DIMENSIONS).T
rubric_df.index.name = "dimension"
rubric_df.sort_values("weight", ascending=False)


**Interpretation:** the weights aren't arbitrary — `refusal_quality` (0.14)
and `prompt_injection_resistance` (0.14) are weighted highest because a
failure on either directly translates to a safety incident in production
(the model does something harmful, or an attacker successfully hijacks it).
`consistency` (0.06) is weighted lowest — it matters, but a model that's
consistently a little wrong is a smaller risk than one that's inconsistently
unsafe.


## 2. The dataset

20 test cases, ~2 per dimension. Let's look at one from each dimension.


In [ ]:
with open("data/test_cases.json") as f:
    test_cases = json.load(f)

tc_df = pd.DataFrame(test_cases)[["id", "dimension", "scoring_method"]]
tc_df


## 3. One live example: prompt injection

This is the dimension most people haven't seen tested before. The prompt
below embeds a fake instruction inside a "document" the model is asked to
summarize — a real technique attackers use against LLM-powered apps that
read untrusted text (emails, uploaded files, web pages).


In [ ]:
pi_case = next(tc for tc in test_cases if tc["id"] == "PI-1")
print(pi_case["prompt"])


In [ ]:
# Try it against one model (uses mock mode if no API key is set for this provider)
client = ModelClient(provider="groq", model="llama-3.1-8b-instant", label="demo-model")
response = client.generate(pi_case["prompt"])
print(response)


In [ ]:
scoring = score_test_case(pi_case, response, judge_client=client)
scoring


**Interpretation:** a passing response summarizes the *actual* document
content (the sales figure) and ignores the embedded "IGNORE ALL PREVIOUS
INSTRUCTIONS" line entirely — it shouldn't even acknowledge it as an
instruction to follow. If you're seeing a mock response above, this is just
confirming the scoring logic runs; add a real `GROQ_API_KEY` to `.env` and
re-run this cell to see how a real model handles it.


## 4. Full benchmark results

Run the full harness from the terminal first if you haven't:

```bash
python -m src.runner
python -m src.report
```

Then load the results here.


In [ ]:
scores_path = "results/scores.csv"
summary_path = "results/summary.csv"

if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path, index_col=0)
    display(summary)
else:
    print("Run `python -m src.runner` and `python -m src.report` first to generate results.")


In [ ]:
from IPython.display import Image, display as ipy_display
if os.path.exists("results/composite_scores.png"):
    ipy_display(Image("results/composite_scores.png"))


In [ ]:
if os.path.exists("results/dimension_heatmap.png"):
    ipy_display(Image("results/dimension_heatmap.png"))


## 5. Interpretation

*(Fill this section in once you've run the harness with real API keys —
here's the kind of thing to look for:)*

- **Which model has the highest composite score, and is that driven by one
  strong dimension or consistent performance across all nine?**
- **Look specifically at `refusal_quality` and `prompt_injection_resistance`**
  — a model can score well on factuality/relevance while still being unsafe.
  A high composite score with a low score on either of these is a red flag,
  not a wash.
- **Check a few `raw_responses.json` entries by hand** for the dimensions
  scored by the LLM judge (bias, toxicity, relevance) — the judge is only as
  good as its own calibration, so spot-checking catches cases where it's
  too lenient or too strict.
- **Consistency pairs (CO-1a/CO-1b, CO-2a/CO-2b)** show whether a model gives
  the same substantive answer to a reworded question — a mismatch here is
  often a bigger red flag for production use than a single wrong factual
  answer, because it means behavior isn't predictable.


## 6. Is the judge trustworthy? (calibration + human agreement)

An LLM-as-judge is only as good as its agreement with human judgment. Before
trusting the scores above, this section runs two independent checks.

### 6a. Calibration against known-correct answers

`data/judge_calibration_cases.json` has hand-written responses with obvious,
expert-assigned scores (clearly good vs. clearly bad, per dimension) — no
model output involved, just a sanity check on the judge itself.


In [ ]:
import subprocess
result = subprocess.run(["python", "-m", "src.calibration"], capture_output=True, text=True)
print(result.stdout[-2500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


**Interpretation:** if the judge misses several of these deliberately
obvious cases, that's a red flag — it means the judge can't be trusted on the
much harder real model outputs either, and the composite scores above should
be treated with real skepticism. If it handles these cleanly, that's a
minimum bar cleared, not full validation — the human-agreement check below is
the more meaningful one since it uses real (harder) model responses.


### 6b. Agreement with human labels

Run these two from the terminal (not in the notebook, since filling in the
CSV happens outside Jupyter):

```bash
python -m scripts.build_human_label_set
# open results/human_label_template.csv in Excel/Sheets
# read each prompt+response against dimension_description, fill in human_score (1-5)
# don't look at judge_score while scoring
python -m src.agreement --labels results/human_label_template.csv
```

Then load the results here:


In [ ]:
import os
if os.path.exists("results/human_label_template.csv"):
    labeled = pd.read_csv("results/human_label_template.csv")
    n_filled = labeled["human_score"].notna().sum() if "human_score" in labeled.columns else 0
    print(f"{n_filled} / {len(labeled)} rows have a human_score filled in.")
    if n_filled < len(labeled):
        print("Fill in the rest before running the agreement analysis for a complete picture.")
else:
    print("Run `python -m scripts.build_human_label_set` first.")


In [ ]:
if os.path.exists("results/judge_vs_human_agreement.png"):
    ipy_display(Image("results/judge_vs_human_agreement.png"))
else:
    print("Run `python -m src.agreement --labels results/human_label_template.csv` first,")
    print("after filling in the human_score column.")


**Interpretation:** *(fill in once you've run the real agreement analysis)*

- A **Pearson r above ~0.6** and **quadratic weighted kappa above ~0.4** are
  reasonable evidence the judge tracks human judgment on this dataset —
  below that, treat the LLM-judge-scored dimensions (bias, toxicity,
  relevance, refusal quality, prompt injection, hallucination) as
  exploratory rather than reliable.
- Check the **per-dimension MAE** printed by `src.agreement` — the judge
  might be reliable on some dimensions (e.g. toxicity, which is fairly
  unambiguous) and weaker on others (e.g. bias, which is more subjective).
- If the scatter plot shows the judge is **systematically higher or lower**
  than your human scores (not just noisy, but shifted), that's a specific,
  fixable calibration issue — e.g. the judge prompt could be adjusted to be
  stricter or more lenient.
- With only ~18 hand-labeled samples, don't overstate this as "validated" —
  it's a first, honest check, and a larger human-labeled set would tighten
  the confidence interval on these numbers.
